In [ ]:
!pip install transformers torch

In [ ]:
from transformers import pipeline

# 1. Tải pipeline "fill-mask"
# Mặc định nó sẽ tải mô hình distilbert-base-uncased (phiên bản nhẹ hơn của BERT)
mask_filler = pipeline("fill-mask")

# 2. Câu đầu vào với token [MASK]
input_sentence = "Hanoi is the (<mask>) of Vietnam."

# 3. Thực hiện dự đoán
# top_k=5: Lấy 5 kết quả có xác suất cao nhất
predictions = mask_filler(input_sentence, top_k=5)

# 4. In kết quả
print(f"Câu gốc: {input_sentence}")
print("-" * 30)
for pred in predictions:
    print(f"Dự đoán: '{pred['token_str']}' | Độ tin cậy: {pred['score']:.4f}")
    print(f" -> Câu hoàn chỉnh: {pred['sequence']}")

No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8 (https://huggingface.co/distilbert/distilroberta-base).
Using a pipeline without specifying a model name and revision in production is not recommended.
Some weights of the model checkpoint at distilbert/distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


Câu gốc: Hanoi is the (<mask>) of Vietnam.
------------------------------
Dự đoán: 'capital' | Độ tin cậy: 0.4033
 -> Câu hoàn chỉnh: Hanoi is the (capital) of Vietnam.
Dự đoán: 'Republic' | Độ tin cậy: 0.3443
 -> Câu hoàn chỉnh: Hanoi is the (Republic) of Vietnam.
Dự đoán: 'north' | Độ tin cậy: 0.0417
 -> Câu hoàn chỉnh: Hanoi is the (north) of Vietnam.
Dự đoán: 'state' | Độ tin cậy: 0.0312
 -> Câu hoàn chỉnh: Hanoi is the (state) of Vietnam.
Dự đoán: 'south' | Độ tin cậy: 0.0246
 -> Câu hoàn chỉnh: Hanoi is the (south) of Vietnam.


In [ ]:
from transformers import pipeline, set_seed

# Đặt seed để kết quả có thể tái lập (tùy chọn)
set_seed(42)

# 1. Tải pipeline "text-generation"
# Mặc định thường là GPT-2
generator = pipeline("text-generation", model="gpt2")

# 2. Đoạn văn bản mồi
prompt = "The best thing about learning NLP is"

# 3. Sinh văn bản
# pad_token_id=50256 để tránh cảnh báo trong thư viện mới
generated_texts = generator(prompt, max_length=50, num_return_sequences=1, pad_token_id=50256)

# 4. In kết quả
print(f"Câu mồi: '{prompt}'")
print("-" * 30)
for text in generated_texts:
    print("Văn bản được sinh ra:")
    print(text['generated_text'])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Câu mồi: 'The best thing about learning NLP is'
------------------------------
Văn bản được sinh ra:
The best thing about learning NLP is that there are a few ways to pick up resources:

Learn something from others.

Learning from teachers.

Learning from friends.

Learning from friends who are knowledgeable and articulate.

Learning from family.

Learning from friends who have had a strong interest in learning NLP.

I wish I had found a good way to give back. I have done this, I have done this, I have done this, I have done this, I have done this, I have done this.

If you're looking for something new, or you look to learn something from other people, then this is the place to start. I encourage people to do this, and to share their experiences with others.

So when you're ready to start learning NLP, this is a great place to start.

What's next?

I'm always looking for the next big thing. I've been in the top 10 (in our top 10 list for NLP learning), my top 10 list for NLP learning, 

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

# 1. Chọn mô hình BERT
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# 2. Câu đầu vào
sentences = ["This is a sample sentence."]

# 3. Tokenize câu
inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')

# 4. Đưa qua mô hình để lấy hidden states
with torch.no_grad():
    outputs = model(**inputs)

# last_hidden_state shape: (batch_size, sequence_length, hidden_size)
last_hidden_state = outputs.last_hidden_state

# 5. Thực hiện Mean Pooling (Tính trung bình cộng)
attention_mask = inputs['attention_mask']

# Mở rộng kích thước mask để khớp với last_hidden_state
# Mask gốc: [batch, seq_len] -> [batch, seq_len, 1] -> [batch, seq_len, hidden_size]
mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()

# Nhân hidden state với mask (để loại bỏ giá trị tại các vị trí padding)
sum_embeddings = torch.sum(last_hidden_state * mask_expanded, 1)

# Tính tổng số lượng token thực (không phải padding) tại mỗi câu
# clamp(min=1e-9) để tránh lỗi chia cho 0
sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)

# Tính trung bình
sentence_embedding = sum_embeddings / sum_mask

# 6. In kết quả
print("Vector biểu diễn của câu (5 giá trị đầu tiên):")
print(sentence_embedding[0][:5])
print("\nKích thước của vector:", sentence_embedding.shape)

Vector biểu diễn của câu (5 giá trị đầu tiên):
tensor([-0.0639, -0.4284, -0.0668, -0.3843, -0.0658])

Kích thước của vector: torch.Size([1, 768])
